1. Le revenu total est la différence entre le chiffre d'affaires et les coûts d'achat des matières premières.
le terme mmin{q,d} : représente la quantité effectivement vendue pour chaque produit. On ne peut pas vendre plus que ce que l'on a fabriqué (q) ni plus que ce que les clients demandent (d).

2. La difficulté de la fonction min{q,d} est qu'elle n'est pas différentiable aux points où q_j = d_j (à écrire en latex). On ne peut donc pas utiliser les méthodes qui nécessitent que la fonction soit C1.

3. On introduit une nouvelle fonction que l'on cherche à maximiser.
Justification de l'approximation (toujours à écrire en latex) : pour alpha>>1
 - si qi<di : e(-alpha qi) >> e(-alpha di) donc hi(q,d) équivaut à qi
 - si qi>di : alors hi équivaut à di
 Intérêt : contrairement à la fonction min, la fonction h est différentiable sur tout son domaine. Donc on peut utiliser des algos d'optimisation plus simples à implémenter.

4. Formulation du problème d'optimisation
 - variables de décision z, nombre n, contraintes c : à faire
 - fonction à minimiser : l'opposée du revenu donc f(r,q) : c_T.r - v_T.h(q,d)

5.  Les méthodes de résolution possibles sont : méthode des points intérieurs, méth de pénalité, algo de gradient projeté... (j'ai pas encore bien lu le cours donc ça se trouve je propose que de la merde)

In [1]:
#question 6.
import numpy as nump
from scipy.optimize import minimize

#paramètres
alpha = 0.1
v = np.array([0.9, 1.5, 1.1])
d = np.array([400, 67, 33])
c = 1e-3 * np.array([30, 1, 1.3, 4, 1])
A = np.array([[3.5, 2, 1], [250, 80, 25], [0, 8, 10], [0, 40, 10], [0, 8.5, 0]])

def h_approche(q, d, alpha):
    return (q * np.exp(-alpha * q) + d * np.exp(-alpha * d)) / (np.exp(-alpha * q) + np.exp(-alpha * d))


NameError: name 'np' is not defined

Question 9.a)

Soit $i \in \{1, \dots, p\}$. On regarde la composante $i$ du produit. 

Sens Direct ($\implies$) : Si on pose $u^k_i = \min(q_i, d^k_i)$, alors par définition du minimum, on a bien $u^k_i \le q_i$ et $u^k_i \le d^k_i$. Dans ce cas, maximiser $v^T \min(q, d^k)$ revient exactement à maximiser $v^T u^k$. 

Sens Réciproque ($\impliedby$) : Supposons que l'on maximise $v_i u^k_i$ sous les contraintes $u^k_i \le q_i$ et $u^k_i \le d^k_i$. Comme le prix de vente $v_i$ est strictement positif ($v_i > 0$), l'optimum sera atteint pour la plus grande valeur possible de $u^k_i$.La plus grande valeur de $u^k_i$ qui respecte simultanément $u^k_i \le q_i$ et $u^k_i \le d^k_i$ est précisément $\min(q_i, d^k_i)$.

Conclusion : Les deux formulations donnent le même maximum et le même point optimal.

Question 9.b)

Question 10.a)
Méthode des sous-gradients

Question 10.b)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Paramètres ---
c = np.array([30, 1, 1.3, 4, 1]) * 1e-3
v = np.array([0.9, 1.5, 1.1])
d_scenarios = [
    np.array([400, 67, 33]),
    np.array([500, 80, 53]),
    np.array([300, 60, 43])
]
pi = np.array([0.5, 0.3, 0.2])
A = np.array([
    [3.5, 2, 1],
    [250, 80, 25],
    [0, 8, 3],
    [0, 40, 10],
    [0, 8.5, 0]
])

# Pré-calcul de c^T * A
cTA = c @ A

def compute_subgradient(q):
    """Calcule le sous-gradient de f par rapport à q"""
    grad_q = np.copy(cTA)
    for k in range(len(d_scenarios)):
        # Indicateur q_i <= d_k_i (sous-gradient du min)
        indicator = (q <= d_scenarios[k]).astype(float)
        grad_q -= pi[k] * v * indicator
    return grad_q

# --- 2. Algorithme du sous-gradient ---
n_iter = 5000
q_k = np.array([400.0, 70.0, 40.0]) # Point de départ
history = []

for k in range(1, n_iter + 1):
    # Pas décroissant pour assurer la convergence (rho_k = a / (b + k))
    rho_k = 1.0 / (10 + 0.1 * k) 
    
    g_k = compute_subgradient(q_k)
    
    # Mise à jour : q = q - rho * g
    q_k = q_k - rho_k * g_k
    
    # Projection sur l'ensemble admissible q >= 0
    q_k = np.maximum(q_k, 0)
    
    # Calcul du profit pour le suivi (on minimise l'opposé)
    profit = -(cTA @ q_k - sum(pi[s] * np.dot(v, np.minimum(q_k, d_scenarios[s])) for s in range(3)))
    history.append(profit)

# --- 3. Résultats ---
print(f"Quantités optimales (q) : {q_k}")
print(f"Matières à acheter (r = Aq) : {A @ q_k}")
print(f"Profit final : {history[-1]:.2f} €")

plt.plot(history)
plt.title("Évolution du profit (Méthode du sous-gradient)")
plt.xlabel("Itérations")
plt.ylabel("Profit (€)")
plt.show()